<div dir="rtl">

# תרגיל 3 — כותבים סימולציה

[פתיחה ב־Colab](https://colab.research.google.com/github/yaminbens/alpha-mentorship/blob/main/notebooks/12_build_simulation.ipynb) · [אתר התלמידים](https://yaminbens.github.io/alpha-mentorship/)

ב־GitHub רואים תצוגה מקדימה. לעריכה ולהרצה פתחו ב־Colab ושמרו עותק ב־Drive.

</div>

<div dir="rtl">

## עותק אישי להמשך העבודה
1. פותחים מאתר התלמידים באמצעות **פתיחה ב־Colab**.
2. בוחרים **File → Save a copy in Drive** ומשנים את השם ל־`קוד-תלמיד_תרגילים`.
3. מריצים תאים לפי הסדר באמצעות המשולש. אחרי שינוי בתא מריצים אותו שוב; תאים שכבר רצו אינם מתעדכנים בעצמם.
4. תשובות כותבים בתאי הטקסט במקום `…`. קוד כותבים בתאים המסומנים **תורכם לכתוב**.
5. כשיש משהו שתרצו להראות או לשאול עליו, שתפו את העותק עם חשבון המנחה ושלחו את הקישור בערוץ הפרטי שסוכם. אפשר לשתף גם עבודה חלקית.

אנימציות וקבצים נוצרים בסביבת ההרצה הזמנית. בחרו ריצות שתרצו לשמור והורידו עבורן את קובצי ה־HTML, ה־PNG, ה־CSV והתנאים דרך סמל התיקייה בצד: שלוש נקודות ליד הקובץ → **Download**. שתפו גם אותם. שמרו גיבוי של המחברת דרך **File → Download → Download .ipynb**. האתר וה־GitHub הציבורי אינם שומרים את התשובות האישיות שלכם.

אם קישור Colab אינו נפתח, הורידו את המחברת מהאתר ופתחו אותה ב־Colab דרך **File → Upload notebook**.

</div>

<div dir="rtl">

## כללי המודל והיחידות
הקופסה היא ריבוע שאורך צלעו 1 **יחידת אורך של המודל**. הזמן נמדד ביחידות זמן של המודל; אלה אינן בהכרח שניות או מטרים. המסה של כל חלקיק היא 1.

בתרגילים האלה החלקיקים **אינם מפעילים כוחות זה על זה** ויכולים לעבור זה דרך זה. הם משנים תנועה רק בגלל קירות או כבידה, לפי התרגיל. זה מודל לימודי של תנועה והתנגשויות; הוא אינו סימולציה של התכה או נוזל אמיתי. הצבע מסמן זהות של קבוצה, לא טמפרטורה.

אנימציה היא אוסף תמונות בזמנים עוקבים. כפתור הניגון ומחוון הזמן מאפשרים להתבונן בתנועה. קצב הצגת התמונות אינו משנה את חישוב התנועה.

</div>

<div dir="rtl">

## הכנת כלי ההדמיה — להריץ פעם אחת
התא הבא מכיל את מנוע התנועה, האנימציה והשמירה. הוא כלול במחברת; אין צורך להתקין ספרייה מה־GitHub או להעתיק קובץ מצב התחלתי. הריצו אותו והמשיכו לתאי התרגילים. אפשר לקרוא אותו בהמשך; אין צורך לשנות אותו כדי להתחיל.

</div>

In [ ]:
"""Small independent-particle teaching models; not an interacting molecular fluid.

Box length = 1 model-length unit, time in model-time units, mass = 1.
Mixing: free motion, elastic walls, a central wall with a configurable opening.
Bouncing: constant downward acceleration, restitution at horizontal walls.
Vertical impacts are resolved at their event times. A rebound below 0.001
model-speed units at the floor is treated as contact to resolve inelastic collapse.
"""
import json
import uuid
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display


def reflect_coordinate(x, v, dt):
    """Elastic motion on [0, 1], including multiple crossings in one step."""
    unfolded = np.asarray(x) + np.asarray(v) * dt
    phase = unfolded % 2.0
    position = np.where(phase <= 1, phase, 2 - phase)
    velocity = np.where(phase < 1, v, -np.asarray(v))
    # Exactly at a wall, choose the outgoing velocity.
    velocity = np.where(position == 0, np.abs(velocity), velocity)
    velocity = np.where(position == 1, -np.abs(velocity), velocity)
    return position, velocity


def vertical_step(y, v, dt, gravity, restitution):
    remaining = float(dt)
    while remaining > 1e-12:
        if gravity > 0 and y <= 1e-12 and 0 <= v < 0.001:
            return 0.0, 0.0
        candidates = []
        for wall in (0.0, 1.0):
            if gravity == 0:
                roots = [(wall-y)/v] if v != 0 else []
            else:
                discriminant = v*v + 2*gravity*(y-wall)
                roots = [] if discriminant < 0 else [(v-np.sqrt(discriminant))/gravity, (v+np.sqrt(discriminant))/gravity]
            for t in roots:
                incoming = v-gravity*t
                if t > 1e-12 and t <= remaining+1e-12 and ((wall == 0 and incoming < 0) or (wall == 1 and incoming > 0)):
                    candidates.append((t,wall))
        # An initial condition already on a wall may point outwards.
        if y <= 1e-12 and v < 0:
            v = -restitution*v
            continue
        if y >= 1-1e-12 and v > 0:
            v = -restitution*v
            continue
        if not candidates:
            return y+v*remaining-0.5*gravity*remaining**2, v-gravity*remaining
        t,wall = min(candidates)
        y = wall
        v = -restitution*(v-gravity*t)
        remaining = max(0.0, remaining-t)
    return y,v


def mixing_run(opening=1.0, speed=0.3, seed=7):
    if not (0 <= opening <= 1 and 0.05 <= speed <= 1):
        raise ValueError('opening בין 0 ל־1; speed בין 0.05 ל־1')
    rng = np.random.default_rng(seed)
    n,dt,steps = 40,0.02,500
    labels = np.repeat([0,1],n//2)
    pos = rng.uniform(0.08,0.92,(n,2))
    pos[:,0] = rng.uniform(0.08,0.42,n)+0.5*labels
    angle = rng.uniform(0,2*np.pi,n)
    vel = speed*np.column_stack((np.cos(angle),np.sin(angle)))
    frames,times = [pos.copy()],[0.0]
    for step in range(1,steps+1):
        old = pos.copy()
        trial = old+vel*dt
        crossed = (old[:,0]-0.5)*(trial[:,0]-0.5) < 0
        for i in np.flatnonzero(crossed):
            fraction = (0.5-old[i,0])/(trial[i,0]-old[i,0])
            y_cross = old[i,1]+fraction*(trial[i,1]-old[i,1])
            if abs(y_cross-0.5) >= opening/2:
                trial[i,0] = 1-trial[i,0]
                vel[i,0] *= -1
        for axis in (0,1):
            pos[:,axis],vel[:,axis] = reflect_coordinate(trial[:,axis],vel[:,axis],0)
        if step % 5 == 0:
            frames.append(pos.copy());times.append(step*dt)
    positions = np.array(frames)
    fraction = (positions[:,labels==0,0]>0.5).mean(axis=1)
    return dict(positions=positions,time=np.array(times),labels=labels,opening=opening,
                measure=fraction,measure_name='Blue fraction on right',
                settings=dict(model='noninteracting mixing',opening=opening,speed=speed,seed=seed,dt=dt,N=n,duration=steps*dt))


def bounce_run(restitution=1.0, gravity=1.0, seed=7):
    if not (0 <= restitution <= 1 and 0 <= gravity <= 3):
        raise ValueError('restitution בין 0 ל־1; gravity בין 0 ל־3')
    rng=np.random.default_rng(seed)
    n,dt,steps=24,0.02,400
    pos=np.column_stack((rng.uniform(.08,.92,n),rng.uniform(.55,.9,n)))
    vel=np.column_stack((rng.uniform(-.15,.15,n),np.zeros(n)))
    frames,times,energy=[pos.copy()],[0.0],[np.mean(0.5*vel[:,1]**2+gravity*pos[:,1])]
    for step in range(1,steps+1):
        pos[:,0],vel[:,0]=reflect_coordinate(pos[:,0],vel[:,0],dt)
        for i in range(n):
            pos[i,1],vel[i,1]=vertical_step(pos[i,1],vel[i,1],dt,gravity,restitution)
        if step % 5 == 0:
            frames.append(pos.copy());times.append(step*dt)
            energy.append(np.mean(0.5*vel[:,1]**2+gravity*pos[:,1]))
    positions=np.array(frames)
    return dict(positions=positions,time=np.array(times),labels=np.arange(n)%2,opening=None,
                measure=positions[:,:,1].mean(axis=1),measure_name='Mean height',energy=np.array(energy),
                settings=dict(model='independent bouncing particles',restitution=restitution,gravity=gravity,seed=seed,dt=dt,N=n,duration=steps*dt,contact_speed=0.001))


def animation_html(result, title='סימולציית חלקיקים'):
    ident='animation_'+uuid.uuid4().hex
    data=json.dumps(dict(positions=np.asarray(result['positions']).round(5).tolist(),
                        time=np.asarray(result['time']).round(3).tolist(),
                        labels=np.asarray(result['labels']).tolist(),opening=result.get('opening')))
    import html
    return '''<!doctype html><html lang="he" dir="rtl"><meta charset="utf-8">
<div id="'''+ident+'''" style="font:16px system-ui;max-width:560px;background:#f5f8fa;padding:16px;color:#203246">
<strong>'''+html.escape(title)+'''</strong><br>
<canvas width="500" height="500" style="width:100%;max-width:500px;background:white" aria-label="תנועת חלקיקים בקופסה"></canvas>
<div><button type="button" class="play">ניגון / עצירה</button> <button type="button" class="reset">להתחלה</button>
<input class="timeline" type="range" min="0" value="0" aria-label="זמן בסימולציה" style="width:45%"><output></output><br><label><input class="trails" type="checkbox"> הצגת מסלולים</label></div>
<p style="font-size:13px">אפשר לגרור את מחוון הזמן. הצבעים מסמנים קבוצות חלקיקים; הם אינם טמפרטורה.</p></div>
<script>(()=>{const root=document.getElementById("'''+ident+'''"),d='''+data+''';
const canvas=root.querySelector('canvas'),ctx=canvas.getContext('2d'),slider=root.querySelector('.timeline'),out=root.querySelector('output'),trails=root.querySelector('.trails');trails.checked=d.positions[0].length===1;
slider.max=d.positions.length-1;let running=false;
function draw(){const k=Number(slider.value);ctx.clearRect(0,0,500,500);ctx.strokeStyle='#26394b';ctx.lineWidth=2;ctx.strokeRect(15,15,470,470);
if(d.opening!==null&&d.opening<1){ctx.beginPath();ctx.moveTo(250,15);ctx.lineTo(250,250-235*d.opening);ctx.moveTo(250,250+235*d.opening);ctx.lineTo(250,485);ctx.stroke();}
if(trails.checked){d.positions[0].forEach((_,i)=>{ctx.strokeStyle=d.labels[i]===0?'#8fbfc4':'#dcb296';ctx.lineWidth=1;ctx.beginPath();for(let j=0;j<=k;j++){const p=d.positions[j][i];if(j===0)ctx.moveTo(15+470*p[0],485-470*p[1]);else ctx.lineTo(15+470*p[0],485-470*p[1]);}ctx.stroke();});}
d.positions[k].forEach((p,i)=>{ctx.fillStyle=d.labels[i]===0?'#087e8b':'#c56732';ctx.beginPath();ctx.arc(15+470*p[0],485-470*p[1],5,0,Math.PI*2);ctx.fill();});out.textContent='זמן: '+d.time[k];}
root.querySelector('.play').onclick=()=>running=!running;root.querySelector('.reset').onclick=()=>{running=false;slider.value=0;draw();};slider.oninput=()=>{running=false;draw();};trails.onchange=draw;
const timer=setInterval(()=>{if(!root.isConnected){clearInterval(timer);return;}if(running){slider.value=(Number(slider.value)+1)%d.positions.length;draw();}},80);draw();})();</script></html>'''


def show_result(result, stem, title):
    """Display and save an animation, static figure, measurements and conditions."""
    document=animation_html(result,title)
    Path(stem+'.html').write_text(document,encoding='utf-8')
    display(HTML(document))
    fig,axes=plt.subplots(1,3,figsize=(10,3.4))
    colors=np.where(np.array(result['labels'])==0,'#087e8b','#c56732')
    for ax,k in zip(axes,[0,len(result['time'])//2,len(result['time'])-1]):
        xy=result['positions'][k];ax.scatter(xy[:,0],xy[:,1],c=colors,s=18)
        if result.get('opening') is not None:
            gap=result['opening'];ax.plot([.5,.5],[0,.5-gap/2],color='gray');ax.plot([.5,.5],[.5+gap/2,1],color='gray')
        ax.set(xlim=(0,1),ylim=(0,1),xlabel='x / model length',ylabel='y / model length',title=f"t = {result['time'][k]:.1f}");ax.set_aspect('equal')
    fig.tight_layout();fig.savefig(stem+'.png',dpi=130);plt.show();plt.close(fig)
    np.savetxt(stem+'.csv',np.column_stack((result['time'],result['measure'])),delimiter=',',header='time,'+result['measure_name'],comments='')
    Path(stem+'_settings.json').write_text(json.dumps(result['settings'],ensure_ascii=False,indent=2),encoding='utf-8')
    print('נשמרו קבצים בשם',stem,': אנימציה html, תמונה png, נתונים csv ותנאים settings.json. הורידו אותם מתיקיית הקבצים ב־Colab.')

<div dir="rtl">

# תרגיל 3 — כותבים סימולציה משלנו
**כ־50–70 דקות.** כאן אתם כותבים את חוקי התנועה והלולאה. נתחיל בחלקיק אחד בתוך ריבוע, ללא כבידה וללא כוחות בין חלקיקים, עם קפיצות אלסטיות מהקירות.

נשתמש ב־x ו־y למיקום, ב־vx ו־vy לרכיבי המהירות, וב־dt למשך צעד הזמן. ההוראות מניחות היכרות עם משתנים, תנאים ולולאות. פונקציה היא קטע קוד שמקבל ערכים ומחזיר תוצאה באמצעות `return`.

### א. משלימים תנועה חופשית
בזמן שאין כוח, המהירות קבועה. במשך זמן dt החלקיק מתקדם מרחק velocity × dt. לכן המיקום החדש הוא המיקום הישן ועוד ההתקדמות. החליפו את `None` בביטוי המתאים. סימן הכפל בפייתון הוא `*`.

`None` הוא מציין מקום, לא מספר. כל עוד הוא קיים, הבדיקות ידווחו מה חסר. אל תחליפו אותו במספר קבוע: הפונקציה צריכה לעבוד עם קלטים שונים.

</div>

<div dir="rtl">

## פונקציה שאפשר להשתמש בה ולבדוק אותה
*כ־5–7 דקות של קריאה וקוד, לפני כתיבת הסימולציה.*

פונקציה מאפשרת לתת שם לחישוב ולהפעיל אותו על קלטים שונים. הפרמטרים שלה מקבלים את ערכי הקלט, ו־`return` מחזיר תוצאה שאפשר לשמור במשתנה או להעביר לחישוב הבא. `print` מציג מידע על המסך; הוא אינו תחליף להחזרת התוצאה.

העתיקו את הדוגמה הבאה לתא חדש. הפונקציה מחשבת ערך כפול, והלולאה משתמשת בה שלוש פעמים:

```python
def twice(number):
    return 2 * number

for value in [1, 3, -2]:
    print(value, twice(value))
```

שנו את הפונקציה כך שתחזיר פי שלושה, ועדכנו גם את שמה. לאחר מכן הוסיפו בדיקה עם קלט שאת תשובתו אתם יודעים. בגרסה המקורית אפשר לכתוב `assert twice(3) == 6`; בגרסה ששיניתם צריך לעדכן את הבדיקה. `assert` עובר בשקט כשהתנאי נכון ומציג שגיאה כשהוא אינו נכון. כך אפשר לבדוק חישוב קטן לפני שמשלבים אותו בתוכנית גדולה.

הסבר נוסף נמצא בסעיף [הגדרת פונקציות במדריך Python](https://docs.python.org/3/tutorial/controlflow.html#defining-functions). התמקדו בפסקה על `return`; אין צורך להבין את אלגוריתם פיבונאצ׳י שבדוגמה. לחזרה על לולאות מספיקים הדוגמה הראשונה בסעיף [for](https://docs.python.org/3/tutorial/controlflow.html#for-statements) והדוגמה `range(5)` בסעיף שאחריו. עצרו לפני הדוגמאות שמשנות אוספים.

שמרו במחברת את הפונקציה ששיניתם, את השימוש בה בלולאה ואת הבדיקה שעוברת. בהמשך תשתמשו באותו מבנה כדי לבדוק תנועה ופגיעה בקיר.

</div>

In [ ]:
def move(position, velocity, dt):
    next_position = None  # TODO: מיקום חדש לפי מיקום, מהירות ומשך הצעד
    return next_position

<div dir="rtl">

### ב. משלימים פגיעה בקיר
הקופסה משתרעת מ־0 עד 1 בכל כיוון. אם המיקום עבר את 1, משקפים את החריגה בחזרה פנימה; למשל 1.1 הופך ל־0.9. משנים את **סימן** רכיב המהירות, בלי לשנות את גודלו. השלימו את שני ה־`None`.

הפונקציה מטפלת בפגיעה אחת לכל ציר בכל צעד. בתרגיל השתמשו ב־`dt = 0.02` וברכיבי מהירות שגודלם לכל היותר 1, כדי שצעד לא יחצה כמה קירות.

</div>

In [ ]:
def wall(position, velocity):
    if position < 0:
        position = -position
        velocity = None  # TODO: הפכו את סימן המהירות
    elif position > 1:
        position = 2 - position
        velocity = None  # TODO: הפכו את סימן המהירות
    return position, velocity

<div dir="rtl">

### ג. משלימים את לולאת הזמן
בכל צעד צריך להזיז את x ואת y באמצעות `move`, ואז לבדוק קירות ולשמור את המיקום. השלימו את שתי הקריאות לפונקציה. `frames.append([x, y])` שומר את המיקום לרשימת התמונות.

הסדר חשוב: מזיזים, מתקנים פגיעה בקיר, ורק אז שומרים. הקוד שמטפל ב־`None` מאפשר להריץ מחברת שעדיין אינה גמורה; הוא לא ממלא את הפתרון במקומכם.

</div>

In [ ]:
def my_simulation(x, y, vx, vy, dt, steps):
    frames = [[x, y]]
    for step in range(steps):
        x = None  # TODO: קראו ל־move עם x, vx ו־dt
        y = None  # TODO: קראו ל־move עם y, vy ו־dt
        if x is None or y is None:
            return None  # משאירים עד להשלמת הקוד: אין עדיין מסלול
        x, vx = wall(x, vx)
        y, vy = wall(y, vy)
        if vx is None or vy is None:
            return None
        frames.append([x, y])
    return np.array(frames)

<div dir="rtl">

### ד. מריצים בדיקות לפני האנימציה
הבדיקות משתמשות במקרים שהתשובה להם ידועה: תנועה בשני הכיוונים, פגיעה בכל קיר, מנוחה ומסלול קצר. אם משהו לא עבר, מתקנים את הפונקציה, מריצים מחדש את התא שלה, ואז את הבדיקות. קראו את שם הבדיקה שנכשלה.

</div>

In [ ]:
def check_student_code():
    checks = [
        ("תנועה ימינה", lambda: np.isclose(move(0.2, 0.3, 0.1), 0.23)),
        ("תנועה שמאלה", lambda: np.isclose(move(0.2, -0.3, 0.1), 0.17)),
        ("פגיעה בקיר ימין", lambda: np.allclose(wall(1.1, 0.3), (0.9, -0.3))),
        ("פגיעה בקיר שמאל", lambda: np.allclose(wall(-0.1, -0.3), (0.1, 0.3))),
        ("ללא פגיעה", lambda: np.allclose(wall(0.4, 0.3), (0.4, 0.3))),
        ("חלקיק נח", lambda: np.allclose(my_simulation(.2,.3,0,0,.02,10), [[.2,.3]]*11)),
        ("מסלול ללא פגיעה", lambda: np.allclose(my_simulation(.2,.3,.1,.2,.1,2), [[.2,.3],[.21,.32],[.22,.34]])),
        ("מסלול עם פגיעה", lambda: np.allclose(my_simulation(.99,.3,1,0,.02,2), [[.99,.3],[.99,.3],[.97,.3]])),
    ]
    passed=0
    for name,test in checks:
        try:
            ok=bool(test())
        except (TypeError, ValueError, NameError):
            ok=False
        print(("עבר: " if ok else "צריך להשלים או לתקן: ")+name)
        passed += int(ok)
    print(f"עברו {passed} מתוך {len(checks)} בדיקות. כל בדיקה שעוברת עוזרת לוודא חלק אחר בתנועה.")
    return passed == len(checks)
student_ready = check_student_code()

<div dir="rtl">

### ה. מפעילים את הסימולציה שכתבתם
אחרי שכל הבדיקות עברו, התא הבא משתמש **בפונקציות שלכם** כדי ליצור את התנועה. כלי העזר מספק רק תצוגה ושמירה.

1. הריצו עם vx=0.3 ו־vy=0.2. האפשרות ״הצגת מסלולים״ מציירת את הדרך שעבר החלקיק; אפשר לכבות ולהדליק אותה.
2. שנו רק vy ל־0. מה השתנה במסלול? שמרו בשם חדש.
3. נסו vx=vy=0.3. הוסיפו שורה שמדפיסה את מספר התמונות באמצעות `len(trajectory)`.
4. אתגר רשות: נסו ליצור מסלול שחוזר קרוב לנקודת ההתחלה. השוו יחסי מהירויות שונים והגדירו מה נחשב ״קרוב״. קראו ל־`my_simulation` בלולאה עבור שתי מהירויות התחלתיות שונות. אל תשנו את הפונקציות כדי להתאים רק לדוגמה אחת.

</div>

In [ ]:
#@title הפעלת הסימולציה שלי
vx = 0.3 #@param {type:"slider", min:-1, max:1, step:0.1}
vy = 0.2 #@param {type:"slider", min:-1, max:1, step:0.1}
run_name = "task3_my_simulation" #@param {type:"string"}
if check_student_code():
    trajectory = my_simulation(0.2, 0.3, vx, vy, 0.02, 400)
    if trajectory is None or not np.isfinite(trajectory).all() or (trajectory < 0).any() or (trajectory > 1).any():
        print("המסלול אינו תקין. בדקו את המיקום ותנאי הקירות.")
    else:
        selected = trajectory[::5]
        own_result = dict(positions=selected[:, None, :], time=np.arange(len(selected))*0.1,
                          labels=np.array([0]), opening=None, measure=selected[:,0], measure_name="x",
                          settings=dict(model="student elastic particle",vx=vx,vy=vy,dt=.02,steps=400))
        show_result(own_result, run_name, "הסימולציה שאני כתבתי")
        # TODO: הוסיפו הדפסת מספר התמונות במסלול המלא
else:
    print("עדיין אין אנימציה: השלימו את הפונקציות והריצו את התאים שלהן מחדש.")

<div dir="rtl">

## הסימולציה שלכם כנקודת פתיחה
שמרו את הגרסה שאליה הגעתם, כולל הקוד, פלט הבדיקות ואנימציה אם נוצרה. אפשר להראות שינוי שנהניתם לעשות, תקלה שנשארה פתוחה או רעיון למסלול נוסף. אין צורך שכל הפונקציות יהיו גמורות כדי לשתף ולהתייעץ.

בדיקות הקוד הן כלי עבודה: הן משוות תנועה למקרים ידועים ועוזרות לאתר טעות. הן אינן ציון. אפשר לחזור אליהן בהדרגה ובעזרת המנחה.

</div>

<div dir="rtl">

## מעבר ממודל תנועה למודל של חומר
*קריאת המשך של כ־8–10 דקות, אחרי תרגיל 3 ובתיאום עם המנחה.*

בתרגילים שלנו חלקיק אינו משפיע על חלקיק אחר. אפשר כבר ליצור תנועה מורכבת בקופסה, אבל חסר מרכיב מרכזי בתיאור חומר: כוחות בין החלקיקים. כוח שתלוי במרחק בין זוג חלקיקים יכול לשנות את המהירות גם כשאין פגיעה בקיר.

סעיף [The Lennard-Jones model במאמר של Schroeder](https://arxiv.org/html/1502.06169v1#S2) מציג מודל כזה. קראו את הפסקה הראשונה, הסתכלו בנוסחה (1) וקראו את ההסבר על r, sigma ו־epsilon מיד אחריה. התמקדו בקשר בין מרחק לאנרגיה; אין צורך לגזור כוח או לחשב חזקות ביד.

הוספת כוחות משנה גם את מבנה התוכנית. לפני שמעדכנים מיקומים צריך לחשב כוחות ולעדכן את המהירויות. כתבו במחברת את הרצף ״חישוב כוחות → עדכון מהירויות → עדכון מיקומים״, וסמנו מה כבר קיים בקוד שלכם ומה צריך להוסיף. הציעו בדיקה למקרה שבו הכוח אפס.

התרשים והבדיקה המוצעת הם נקודת מוצא לשיחה על הפרויקט הבא. הם מתארים הרחבה שעדיין צריך לממש ולבדוק, ולא תוצאה שהמודל הנוכחי כבר השיג.

</div>

<div dir="rtl">

# לקראת השיחה שלנו

העבודה במחברת היא הזדמנות לבדוק מה מושך אתכם בסימולציות ובתכנות. נשמח לשמוע איך הייתה החוויה שלכם, גם אם הספקתם לחקור רק חלק מהתרגיל.

אפשר לכתוב בכמה שורות על משהו שסיקרן או הפתיע אתכם, על חלק שהיה קל וזרם, או על מקום שבו נתקעתם. אם התעורר רעיון לעוד ניסוי, לכלי שתרצו ללמוד או לכיוון אחר שמעניין אתכם, הוסיפו אותו. אין צורך להתייחס לכל הנקודות; בחרו את מה שחשוב לכם לספר.

אפשר גם לצרף אנימציה, קטע קוד או צילום של תקלה שתרצו להראות ולדבר עליהם. קוד חלקי ותוצאה לא צפויה יכולים להיות התחלה טובה לשיחה.

## כמה מילים שלי

…

## משהו שבחרתי להראות — לבחירה

…

</div>